# Data Cleaning - Ames Housing

Praktikum Data Mining & Data Cleansing

Notebook ini memuat latihan pembersihan data sebelum tahap pemodelan machine learning. Fokus utama mencakup: transformasi log, penanganan duplikat, pengisian missing values, feature scaling, dan deteksi outlier. Dataset yang digunakan adalah data penjualan rumah di Ames, Iowa (2006-2010) dengan target `SalePrice`.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_style('whitegrid')

## 2. Data Reading

In [ ]:
PATH = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/Ames_Housing_Data1.tsv'
raw = pd.read_csv(PATH, sep='\t')
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
# Membuat salinan data untuk proses pembersihan
df = raw.copy()
df['SalePrice'].describe()

### Latihan 1: Analisis Kategori pada `Sale Condition`

In [ ]:
kondisi = df['Sale Condition'].value_counts()
persen = df['Sale Condition'].value_counts(normalize=True).mul(100).round(2)
pd.concat([kondisi, persen], axis=1, keys=['jumlah', 'persen'])

## 3. Korelasi Fitur dengan SalePrice

In [ ]:
num_cols = df.select_dtypes(include='number')
korelasi = num_cols.corr()['SalePrice'].drop('SalePrice')
kuat = korelasi[korelasi.abs() > 0.5].sort_values(ascending=False)

print(f'{len(kuat)} fitur dengan korelasi kuat terhadap SalePrice')
kuat

In [ ]:
fitur_top = kuat.index[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, f in zip(axes, fitur_top):
    ax.scatter(df[f], df['SalePrice'], s=8, alpha=0.5)
    ax.set_xlabel(f)
axes[0].set_ylabel('SalePrice')
plt.tight_layout()
plt.show()

## 4. Transformasi Log

In [ ]:
harga_log = np.log(df['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['SalePrice'], kde=True, ax=axes[0])
axes[0].set_title('sebelum log')
sns.histplot(harga_log, kde=True, ax=axes[1])
axes[1].set_title('sesudah log')
plt.show()

print('skewness sebelum:', round(df['SalePrice'].skew(), 3))
print('skewness sesudah:', round(harga_log.skew(), 3))

### Latihan 2: Transformasi pada `Lot Area`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Lot Area'], kde=True, ax=axes[0])
axes[0].set_title('Lot Area (asli)')
sns.histplot(np.log(df['Lot Area']), kde=True, ax=axes[1])
axes[1].set_title('Lot Area (log)')
plt.show()

print('skewness asli:', round(df['Lot Area'].skew(), 3))
print('skewness log :', round(np.log(df['Lot Area']).skew(), 3))

## 5. Penanganan Duplikat

In [ ]:
print('jumlah PID dobel:', df['PID'].duplicated().sum())
sebelum = len(df)
df = df.drop_duplicates(subset='PID', keep='first')
print(f'{sebelum} -> {len(df)} baris')

### Latihan 3: Hapus Duplikat pada Kolom `Order`

In [ ]:
tes = df.drop_duplicates(subset=['Order'])
print('sebelum:', len(df), '| sesudah:', len(tes))

## 6. Penanganan Missing Values

In [ ]:
hilang = df.isnull().sum()
hilang = hilang[hilang > 0].sort_values(ascending=False)
tabel = pd.DataFrame({'jumlah': hilang, 'persen': (hilang / len(df) * 100).round(1)})
tabel.head(15)

In [ ]:
# Imputasi median pada Lot Frontage
median_frontage = df['Lot Frontage'].median()
df['Lot Frontage'] = df['Lot Frontage'].fillna(median_frontage)

### Latihan 4: Imputasi Mean pada `Mas Vnr Area`

In [ ]:
mean_mas = df['Mas Vnr Area'].mean()
df['Mas Vnr Area'] = df['Mas Vnr Area'].fillna(mean_mas)
print('Sisa null di Mas Vnr Area:', df['Mas Vnr Area'].isnull().sum())

## 7. Feature Scaling & Latihan 5 (`SalePrice`)

In [ ]:
scaler = StandardScaler()
scaled_sprice = scaler.fit_transform(df[['SalePrice']])
print('Bentuk hasil standarisasi:', scaled_sprice.shape)
scaled_sprice[:5]

## 8. Deteksi & Hapus Outlier (Latihan 6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['Lot Area'], ax=axes[0])
axes[0].set_title('Boxplot Lot Area')
axes[1].scatter(df['Lot Area'], df['SalePrice'], s=8, alpha=0.5)
axes[1].set_xlabel('Lot Area')
axes[1].set_ylabel('SalePrice')
plt.tight_layout()
plt.show()

df['Lot_Area_Zscore'] = stats.zscore(df['Lot Area'])
print(df[['Lot Area', 'Lot_Area_Zscore']].describe().round(3))

# Menghapus 1 outlier terekstrem
idx_outlier = df.sort_values(by='Lot Area', ascending=False).index[:1]
df_clean = df.drop(idx_outlier)
print(f'Jumlah baris setelah pembersihan outlier: {len(df_clean)}')